In [1]:
import pandas as pd
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [2]:
df = pd.read_csv("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")
print(df.head())
print(df["sentiment"].value_counts())


                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [3]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["review"] = df["review"].apply(clean_text)


In [4]:
label_map = {"negative": 0, "positive": 1}
df["label"] = df["sentiment"].map(label_map)


In [5]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["review"].values,
    df["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=df["label"].values
)


In [6]:
def tokenize(text):
    return text.split()


In [7]:
from collections import Counter

def tokenize(text):
    return text.split()

def build_vocab(texts, max_vocab=20000, min_freq=2):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))

    words = [w for w, c in counter.items() if c >= min_freq]
    words = sorted(words, key=lambda w: counter[w], reverse=True)
    words = words[:max_vocab]

    stoi = {"<pad>": 0, "<unk>": 1}
    for i, w in enumerate(words, start=2):
        stoi[w] = i

    itos = {i: w for w, i in stoi.items()}
    return stoi, itos


MAX_VOCAB = 20000
stoi, itos = build_vocab(train_texts, max_vocab=MAX_VOCAB, min_freq=2)

vocab_size = len(stoi)
PAD_IDX = stoi["<pad>"]
UNK_IDX = stoi["<unk>"]

print("Vocab size:", vocab_size)


Vocab size: 20002


In [8]:
MAX_LEN = 200

def text_to_ids(text, stoi, max_len=200):
    tokens = tokenize(text)
    ids = [stoi.get(tok, UNK_IDX) for tok in tokens]

    # truncate
    ids = ids[:max_len]

    # pad
    if len(ids) < max_len:
        ids += [PAD_IDX] * (max_len - len(ids))

    return ids


In [9]:
import torch
from torch.utils.data import Dataset, DataLoader

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, stoi, max_len=200):
        self.texts = texts
        self.labels = labels
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = text_to_ids(self.texts[idx], self.stoi, self.max_len)
        y = self.labels[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


BATCH_SIZE = 64

train_dataset = IMDBDataset(train_texts, train_labels, stoi, max_len=MAX_LEN)
test_dataset  = IMDBDataset(test_texts, test_labels, stoi, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [10]:
import torch.nn as nn

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=2, pad_idx=0):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        self.rnn = nn.RNN(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)      # (batch, seq_len, embed_dim)
        out, _ = self.rnn(emb)       # (batch, seq_len, hidden_dim)

        last_out = out[:, -1, :]     # (batch, hidden_dim)
        logits = self.fc(last_out)   # (batch, 2)

        return logits


In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SentimentRNN(
    vocab_size=vocab_size,
    embed_dim=128,
    hidden_dim=128,
    num_classes=2,
    pad_idx=PAD_IDX
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [12]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = torch.argmax(logits, dim=1)

        correct += (preds == yb).sum().item()
        total += xb.size(0)

    return total_loss / total, correct / total


In [13]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        preds = torch.argmax(logits, dim=1)

        correct += (preds == yb).sum().item()
        total += xb.size(0)

    return total_loss / total, correct / total


In [ ]:
EPOCHS = 10

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    test_loss, test_acc = evaluate(model, test_loader)

    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Test  Acc: {test_acc:.4f}")
    print("-" * 50)


Epoch 1/10
Train Loss: 0.6915 | Train Acc: 0.5226
Test  Loss: 0.6962 | Test  Acc: 0.5086
--------------------------------------------------
Epoch 2/10
Train Loss: 0.6880 | Train Acc: 0.5311
Test  Loss: 0.7068 | Test  Acc: 0.5189
--------------------------------------------------
Epoch 3/10
Train Loss: 0.6850 | Train Acc: 0.5351
Test  Loss: 0.7161 | Test  Acc: 0.4975
--------------------------------------------------


In [17]:
@torch.no_grad()
def predict_sentiment(text):
    model.eval()

    # (assuming you already cleaned df texts before train)
    text = text.lower()

    ids = text_to_ids(text, stoi, max_len=MAX_LEN)
    x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)

    logits = model(x)
    probs = torch.softmax(logits, dim=1)   # ✅ softmax output

    pred_id = torch.argmax(probs, dim=1).item()

    id_to_label = {0: "NEGATIVE", 1: "POSITIVE"}
    return id_to_label[pred_id], probs[0][pred_id].item()


review = input("Enter a movie review: ")
pred, conf = predict_sentiment(review)

print("Prediction:", pred)
print("Confidence:", conf)


Enter a movie review: it was average
Prediction: NEGATIVE
Confidence: 0.5116847157478333
